# How SPLIT is computed

This notebook walks through the computation of the **Spatial Preference in Local Interaction (SPLIT)** on a small hypothetical dataset.

SPLIT compares where residents of a tract actually go (**S_obs**) with where they could go (**S_exp**):

$$\text{SPLIT}_i(s) = \frac{S^{obs}_i(s) - S^{exp}_i(s)}{1 - S^{exp}_i(s)}$$

Positive values mean residents visit tracts that are more similar to their own than the nearby tracts available to them; negative values mean they visit tracts that are less similar.

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.append("..")
from split import compute_split, cosine_similarity_matrix

radii = [10, 50, 100]

## 1. Inputs

**Tracts**: centroid coordinates (km) and composition shares. Here race/ethnicity has three groups (`race_a`, `race_b`, `race_c`) and income three ranges (`inc_low`, `inc_mid`, `inc_high`). The real data uses census racial/ethnic shares and five household-income ranges.

In [2]:
tracts = pd.DataFrame(
    {
        "x": [0, 6, 0, 30, 40, 90],
        "y": [0, 0, 9, 0, 28, 0],
        "race_a": [0.8, 0.7, 0.1, 0.2, 0.6, 0.3],
        "race_b": [0.1, 0.2, 0.8, 0.2, 0.3, 0.3],
        "race_c": [0.1, 0.1, 0.1, 0.6, 0.1, 0.4],
        "inc_low": [0.2, 0.3, 0.6, 0.4, 0.1, 0.3],
        "inc_mid": [0.5, 0.5, 0.3, 0.4, 0.4, 0.4],
        "inc_high": [0.3, 0.2, 0.1, 0.2, 0.5, 0.3],
    },
    index=pd.Index(list("ABCDEF"), name="GEOID"),
)
attributes = {"race": ["race_a", "race_b", "race_c"],
              "income": ["inc_low", "inc_mid", "inc_high"]}
tracts

,x,y,race_a,race_b,race_c,inc_low,inc_mid,inc_high
GEOID,,,,,,,,
A,0,0,0.8,0.1,0.1,0.2,0.5,0.3
B,6,0,0.7,0.2,0.1,0.3,0.5,0.2
C,0,9,0.1,0.8,0.1,0.6,0.3,0.1
D,30,0,0.2,0.2,0.6,0.4,0.4,0.2
E,40,28,0.6,0.3,0.1,0.1,0.4,0.5
F,90,0,0.3,0.3,0.4,0.3,0.4,0.3


**Flows**: annual OD flows, expressed as the **share** of each origin's total outflows, so the shares of one origin sum to 1. A row with `Origin == Destination` is a self-loop (trips within the home tract). Tract F has no outflows.

(In the dataset, `Frequency` is the population-rescaled number of visits. Only relative values within an origin matter, so shares give the same results.)

In [3]:
flows = pd.DataFrame(
    [("A", "A", 0.50), ("A", "B", 0.30), ("A", "C", 0.05), ("A", "E", 0.15),
     ("B", "B", 0.40), ("B", "A", 0.40), ("B", "D", 0.20),
     ("C", "C", 0.60), ("C", "A", 0.30), ("C", "F", 0.10),
     ("D", "D", 0.50), ("D", "B", 0.25), ("D", "E", 0.25),
     ("E", "E", 0.40), ("E", "A", 0.45), ("E", "D", 0.15)],
    columns=["Origin", "Destination", "Frequency"],
)
flows.pivot(index="Origin", columns="Destination", values="Frequency").fillna(0)

Destination,A,B,C,D,E,F
Origin,,,,,,
A,0.50,0.30,0.05,0.00,0.15,0.0
B,0.40,0.40,0.00,0.20,0.00,0.0
C,0.30,0.00,0.60,0.00,0.00,0.1
D,0.00,0.25,0.00,0.50,0.25,0.0
E,0.45,0.00,0.00,0.15,0.40,0.0


## 2. Buffers B_s(i)

`B_s(i)` is the set of **other** tracts whose centroid lies within *s* km of tract *i*. Distances between centroids (km):

In [4]:
xy = tracts[["x", "y"]].to_numpy()
dist = pd.DataFrame(np.linalg.norm(xy[:, None] - xy[None], axis=-1),
                    index=tracts.index, columns=tracts.index)
dist.round(1)

GEOID,A,B,C,D,E,F
GEOID,,,,,,
A,0.0,6.0,9.0,30.0,48.8,90.0
B,6.0,0.0,10.8,24.0,44.0,84.0
C,9.0,10.8,0.0,31.3,44.3,90.4
D,30.0,24.0,31.3,0.0,29.7,60.0
E,48.8,44.0,44.3,29.7,0.0,57.3
F,90.0,84.0,90.4,60.0,57.3,0.0


Tracts in each buffer, for *s* = 10, 50 and 100 km:

- **10 km**: only A, B and C are close together. D, E and F have no other tract within 10 km.
- **50 km**: A to E are all within 50 km of each other. F (57 km from its nearest tract, E) is still alone.
- **100 km**: every tract is in every other tract's buffer.

In [5]:
buffers = {s: {g: [h for h in tracts.index if h != g and dist.loc[g, h] <= s] for g in tracts.index}
           for s in radii}
pd.DataFrame({f"B_{s}": {g: ", ".join(b) or "-" for g, b in buffers[s].items()} for s in radii})

,B_10,B_50,B_100
A,"B, C","B, C, D, E","B, C, D, E, F"
B,A,"A, C, D, E","A, C, D, E, F"
C,A,"A, B, D, E","A, B, D, E, F"
D,-,"A, B, C, E","A, B, C, E, F"
E,-,"A, B, C, D","A, B, C, D, F"
F,-,-,"A, B, C, D, E"


## 3. Outflow share

From here on we follow tract **A**. The outflow share is the share of A's outflows going to other tracts in the buffer. Because flows are already shares, this is simply the sum of the shares to tracts in the buffer (the self-loop is never counted).

In [6]:
geoid = "A"
a_out = flows[flows["Origin"] == geoid].set_index("Destination")["Frequency"]

table = pd.DataFrame({"share": a_out, "distance_km": dist.loc[geoid, a_out.index].round(1)})
for s in radii:
    table[f"in_B_{s}"] = table.index.isin(buffers[s][geoid])
display(table)

out_share = {s: a_out[a_out.index.isin(buffers[s][geoid])].sum() for s in radii}
pd.Series(out_share, name="out_share").rename_axis("s (km)").round(3)

,share,distance_km,in_B_10,in_B_50,in_B_100
Destination,,,,,
A,0.50,0.0,False,False,False
B,0.30,6.0,True,True,True
C,0.05,9.0,True,True,True
E,0.15,48.8,False,True,True


s (km)
10     0.35
50     0.50
100    0.50
Name: out_share, dtype: float64

At 10 km only the flows to B and C count. At 50 km the flow to E (49 km away) is added. At 100 km nothing changes: F is added to the buffer but A sends no flow there.

## 4. Cosine similarity between compositions

The similarity of two tracts is the cosine similarity of their composition vectors: 1 for identical compositions, lower for more different ones. We use race/ethnicity here; income works the same way.

In [7]:
race = tracts[attributes["race"]].to_numpy()
sim = pd.DataFrame(cosine_similarity_matrix(race), index=tracts.index, columns=tracts.index)
sim.round(3)

GEOID,A,B,C,D,E,F
GEOID,,,,,,
A,1.000,0.988,0.258,0.445,0.944,0.654
B,0.988,1.000,0.402,0.492,0.983,0.723
C,0.258,0.402,1.000,0.445,0.563,0.654
D,0.445,0.492,0.445,1.000,0.533,0.931
E,0.944,0.983,0.563,0.533,1.000,0.784
F,0.654,0.723,0.654,0.931,0.784,1.000


## 5. Unadjusted SPLIT (S_obs)

Mean similarity between A and its destinations in the buffer, **weighted by outflow** (the weights are renormalised to sum to 1 within the buffer). This reflects where residents of A actually go.

In [8]:
s_obs = {}
for s in radii:
    w = a_out[a_out.index.isin(buffers[s][geoid])]
    w = w / w.sum()
    s_obs[s] = (w * sim.loc[geoid, w.index]).sum()
    terms = " + ".join(f"{w[d]:.3f}×{sim.loc[geoid, d]:.3f}" for d in w.index)
    print(f"s = {s:>3} km: S_obs = {terms} = {s_obs[s]:.3f}")

s =  10 km: S_obs = 0.857×0.988 + 0.143×0.258 = 0.884
s =  50 km: S_obs = 0.600×0.988 + 0.100×0.258 + 0.300×0.944 = 0.902
s = 100 km: S_obs = 0.600×0.988 + 0.100×0.258 + 0.300×0.944 = 0.902


## 6. Structural similarity (S_exp)

**Unweighted** mean similarity between A and every tract in the buffer, whether or not anyone travels there. This is the similarity expected if destinations were chosen at random within the buffer, and it does not depend on mobility.

In [9]:
s_exp = {}
for s in radii:
    b = buffers[s][geoid]
    s_exp[s] = sim.loc[geoid, b].mean()
    terms = ", ".join(f"{h}:{sim.loc[geoid, h]:.3f}" for h in b)
    print(f"s = {s:>3} km: S_exp = mean({terms}) = {s_exp[s]:.3f}")

s =  10 km: S_exp = mean(B:0.988, C:0.258) = 0.623
s =  50 km: S_exp = mean(B:0.988, C:0.258, D:0.445, E:0.944) = 0.659
s = 100 km: S_exp = mean(B:0.988, C:0.258, D:0.445, E:0.944, F:0.654) = 0.658


## 7. SPLIT

SPLIT rescales the gap between observed and expected similarity by the room left above S_exp, so 1 means residents only visit tracts identical to their own. When S_exp ≥ 0.9999 every available tract is (almost) identical, there is no alternative to choose, and SPLIT is set to 0.

In [10]:
split = {s: 0.0 if s_exp[s] >= 0.9999 else (s_obs[s] - s_exp[s]) / (1 - s_exp[s]) for s in radii}
summary = pd.DataFrame({"out_share": out_share, "S_obs": s_obs, "S_exp": s_exp, "SPLIT": split})
summary.rename_axis("s (km)").round(3)

,out_share,S_obs,S_exp,SPLIT
s (km),,,,
10,0.35,0.884,0.623,0.692
50,0.50,0.902,0.659,0.712
100,0.50,0.902,0.658,0.713


- **10 km**: A sends far more to B (similar) than to C (different), so S_obs is well above S_exp and SPLIT is positive.
- **50 km**: D, which is very different from A, enters the buffer but receives no flow from A. S_exp changes little because E, which is similar to A, enters too. E receives flow, so S_obs rises slightly.
- **100 km**: F enters the buffer but A sends nothing there, so S_obs is unchanged and only S_exp moves.

## 8. All tracts and radii with `compute_split`

`compute_split` does the same for every tract, radius and attribute at once. The values for A match the hand computation above. Values that cannot be computed are treated as NaN.

In [11]:
result = compute_split(flows, tracts, attributes, radii_km=radii)
result = result.replace(-1, np.nan)  # treat undefined values as NaN
result

,out_shareP_10,out_shareP_50,out_shareP_100,unadjusted_race_10,unadjusted_race_50,unadjusted_race_100,unadjusted_income_10,unadjusted_income_50,unadjusted_income_100,str_sim_race_10,...,str_sim_race_100,str_sim_income_10,str_sim_income_50,str_sim_income_100,SPLIT_Race_10,SPLIT_Race_50,SPLIT_Race_100,SPLIT_Income_10,SPLIT_Income_50,SPLIT_Income_100
GEOID,,,,,,,,,,,,,,,,,,,,,
A,0.35,0.5,0.5,0.883900,0.901852,0.901852,0.937093,0.933813,0.933813,0.622931,...,0.657875,0.845616,0.884161,0.902074,0.692098,0.712394,0.713122,5.925311e-01,0.428627,0.324109
B,0.40,0.6,0.6,0.988287,0.822980,0.822980,0.973684,0.973566,0.973566,0.988287,...,0.717859,0.973684,0.902546,0.916782,0.000000,0.375691,0.372583,4.218847e-15,0.728750,0.682348
C,0.30,0.3,0.4,0.257576,0.257576,0.356784,0.717547,0.717547,0.746771,0.257576,...,0.464396,0.717547,0.752938,0.769239,0.000000,-0.273219,-0.200915,0.000000e+00,-0.143247,-0.097365
D,0.00,0.5,0.5,NaN,0.512916,0.512916,NaN,0.872423,0.872423,NaN,...,0.569462,NaN,0.899475,0.913945,NaN,0.064848,-0.131340,NaN,-0.269108,-0.482507
E,0.00,0.6,0.6,NaN,0.841171,0.841171,NaN,0.887498,0.887498,NaN,...,0.761368,NaN,0.761744,0.789342,NaN,0.349749,0.334421,NaN,0.527810,0.465949
F,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.749385,NaN,NaN,0.930691,NaN,NaN,NaN,NaN,NaN,NaN
